# A statistical study: magnetosheath density vs IMF Bz

How do the magnetosheath ion density, relative to the upstream solar-wind density, and the
magnetic field strength vary between the magnetopause and the bow shock, and does the answer
depend on the sign of the upstream IMF Bz?

Each MANGO magnetosheath row carries the paired upstream conditions (`Np_sw`, `Bz_imf`) and a
normalized position `R_norm = (|r| - R_mp) / (R_bs - R_mp)`, which is 0 at the magnetopause and
1 at the bow shock along the spacecraft direction. Samples taken under different solar-wind
conditions can therefore be stacked in one statistical profile.

```{note}
These pages run on a small real-data sample served locally (dataset version
`2026.0-docs-sample`), dominated by two days of THEMIS-A data. The figures illustrate the
method; they are not a scientific result.
```

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import polars as pl

import space_mango as mango


`normalized_only=True` keeps the rows that have a normalized position (`R_norm`,
`*_gsm_norm`); `sw_paired_only=True` keeps the rows paired with an upstream solar-wind sample,
so that `Np_sw` and `Bz_imf` are defined. Count first: nothing is downloaded.

In [ ]:
columns = ["Np", "Bx", "By", "Bz", "Np_sw", "Bz_imf", "R_norm", "X_gsm_norm", "Y_gsm_norm"]
query = dict(columns=columns, normalized_only=True, sw_paired_only=True)
mango.count("magnetosheath", **query)

This query spans the whole magnetosheath region. Check `download_mb_estimate` before
downloading: on the full dataset (public server) it is much larger than on this sample, so add
`spacecraft=`, `start=` and `stop=` to `query` (for example one spacecraft over one year) until
the estimate suits you.

In [ ]:
r = mango.magnetosheath.get_data(**query)
df = r.to_polars().with_columns(
    B=(pl.col("Bx") ** 2 + pl.col("By") ** 2 + pl.col("Bz") ** 2).sqrt(),
    n_ratio=pl.col("Np") / pl.col("Np_sw"),
)
df

## Profiles across the magnetosheath

`R_norm` is split into 10 bins between 0 and 1. In each bin, and separately for `Bz_imf < 0`
and `Bz_imf > 0`, the line is the median and the shaded band the interquartile range (25th to
75th percentile). Rows with `R_norm` outside [0, 1] or with `Bz_imf = 0` are left out; an
empty bin leaves a gap in the curve.

In [ ]:
meta = r.metadata
edges = np.linspace(0, 1, 11)
centers = 0.5 * (edges[:-1] + edges[1:])


def binned_quartiles(d, column):
    """25th, 50th and 75th percentiles of `column` in each R_norm bin (NaN where empty)."""
    bin_index = np.digitize(d["R_norm"].to_numpy(), edges) - 1
    values = d[column].to_numpy()
    out = np.full((3, len(centers)), np.nan)
    for i in range(len(centers)):
        in_bin = values[bin_index == i]
        if in_bin.size:
            out[:, i] = np.percentile(in_bin, [25, 50, 75])
    return out


bz_unit = meta["Bz_imf"]["unit"]
imf = {
    (f"Bz_imf < 0 {bz_unit}", "tab:blue"): df.filter(pl.col("Bz_imf") < 0),
    (f"Bz_imf > 0 {bz_unit}", "tab:red"): df.filter(pl.col("Bz_imf") > 0),
}
panels = {
    "n_ratio": "Np / Np_sw (dimensionless)",
    "B": f"|B| ({meta['Bx']['unit']})",
}

fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharex=True)
for ax, (column, ylabel) in zip(axes, panels.items(), strict=True):
    for (name, color), d in imf.items():
        q25, q50, q75 = binned_quartiles(d, column)
        ax.plot(centers, q50, "o-", color=color, label=name)
        ax.fill_between(centers, q25, q75, color=color, alpha=0.2)
    ax.set_xlim(0, 1)
    ax.set_xlabel("R_norm (0: magnetopause, 1: bow shock)")
    ax.set_ylabel(ylabel)
    ax.legend()
fig.suptitle("Median (line) and interquartile range (shading) in each R_norm bin")
fig.tight_layout()
plt.show()

## Where were these samples?

`X_gsm_norm` and `Y_gsm_norm` are GSM positions radially rescaled between fixed average
boundaries, so samples taken when the magnetopause and bow shock were at different distances
can be overlaid on one map. The map below is a projection on the GSM X-Y plane (Z is ignored);
each hexagon shows the median `Np / Np_sw` of the samples it contains (at least 5).

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
hexes = ax.hexbin(
    df["X_gsm_norm"].to_numpy(),
    df["Y_gsm_norm"].to_numpy(),
    C=df["n_ratio"].to_numpy(),
    reduce_C_function=np.median,
    gridsize=30,
    mincnt=5,
)
fig.colorbar(hexes, ax=ax, label="median Np / Np_sw (dimensionless)", shrink=0.8)
ax.plot(0, 0, "o", color="black", label="Earth")
ax.set_aspect("equal")
ax.set_xlabel(f"X_gsm_norm ({meta['X_gsm_norm']['unit']}), Sun towards +X")
ax.set_ylabel(f"Y_gsm_norm ({meta['Y_gsm_norm']['unit']})")
ax.legend(loc="center left")
fig.tight_layout()
plt.show()

## On the full dataset

These pages point `space_mango` at a local server on the sample through the
`SPACE_MANGO_URL` environment variable. In your own session, leave `SPACE_MANGO_URL` unset and
the same code queries the public MANGO server. Run the `mango.count(...)` cell first and narrow
the query until `download_mb_estimate` is acceptable, for example:

```python
query = dict(columns=columns, normalized_only=True, sw_paired_only=True,
             spacecraft="THA", start="2008-01-01", stop="2009-01-01")
mango.count("magnetosheath", **query)
```

Downloads are cached on disk, so running the notebook again does not download the same data
twice (see the user guide).